In [3]:
def process_excel(file_path):
    """加载 Excel 文件，提取分词后的内容，并根据空格拆分生成三元组"""
    # 加载 Excel 文件
    df = pd.read_excel(file_path)
    
    # 只提取 COL_C 和 segmented_text 列
    df_subset = df[['COL_C', 'segmented_text']]

    # 新的列表来保存处理后的数据
    new_rows = []

    # 遍历 DataFrame 中的每一行
    for index, row in df_subset.iterrows():
        col_c_value = row['COL_C']
        
        # 确保 segmented_text 是字符串类型，并处理 NaN 或其他类型
        segmented_text = str(row['segmented_text']) if pd.notna(row['segmented_text']) else ""
        
        # 将 segmented_text 按空格拆分成多个词
        words = segmented_text.split(' ') if segmented_text else []
        
        # 为每个词创建一行新的三元组
        for word in words:
            new_rows.append([col_c_value, word])  # COL_C 和 单个词组合成一行

    # 将新生成的行转换为 DataFrame
    new_df = pd.DataFrame(new_rows, columns=['COL_C', 'Element'])

    # 输出为新的 Excel 文件
    output_path = file_path.replace('.xlsx', '_processed.xlsx')
    new_df.to_excel(output_path, index=False)
    print(f"处理后的数据已保存到 {output_path}")

# 示例使用
file_path = r"E:\00people's commune_only_production.xlsx"  # 替换为你的文件路径
process_excel(file_path)

处理后的数据已保存到 E:\00people's commune_only_production_processed.xlsx


In [4]:
import pandas as pd
import requests
from tqdm import tqdm  # 导入 tqdm
import requests
import pandas as pd
import time
from typing import Tuple
import json

In [ ]:
# 配置DeepSeek API
API_KEY = "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # 替换为您的实际API密钥
API_URL = "https://api.deepseek.com/chat/completions"

In [6]:
def classify_word(word):
    """使用 DeepSeek API 对单个词进行产业分类"""
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    # 分类提示词
    prompt = f"""
    你是一个专业的经济活动分类器。请将以下关于村落居民生产活动的词语分类：
    1. **农业**：直接涉及农作物种植、畜禽养殖、林产品采集等第一产业活动
    2. **工业**：涉及产品制造、加工、维修等第二产业活动
    3. **其他**：不属于以上两类的服务业或其他活动
    
    分类规则：
    - 农业类：粮食作物(如米、麦)、经济作物、畜禽养殖(养猪、养牛)、林产品(人参、药材)
    - 工业类：工厂生产(农具工厂)、设备维修(修配厂)、手工艺品制作
    - 其他：商业(小卖部)、旅游、教育、渔业(捕鱼)、运输等

    请只返回分类结果：农业、工业、其他
    需要分类的词语：{word}
    """

    payload = {
        "model": "deepseek-chat",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 10
    }

    try:
        response = requests.post(API_URL, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()["choices"][0]["message"]["content"].strip()
        
        if "农业" in result:
            return "农业"
        elif "工业" in result:
            return "工业"
        else:
            return "其他"
    
    except Exception as e:
        print(f"分类错误: {word} - {str(e)}")
        return "分类失败"

def process_excel(file_path):
    """加载 Excel 文件，拆分分词并进行产业分类"""
    # 加载 Excel 文件
    df = pd.read_excel(file_path)
    
    # 只提取 COL_C 和 segmented_text 列
    df_subset = df[['COL_C', 'segmented_text']]

    # 新的列表来保存处理后的数据
    new_rows = []

    # 遍历 DataFrame 中的每一行
    for index, row in df_subset.iterrows():
        col_c_value = row['COL_C']
        
        # 确保 segmented_text 是字符串类型，并处理 NaN 或其他类型
        segmented_text = str(row['segmented_text']) if pd.notna(row['segmented_text']) else ""
        
        # 将 segmented_text 按空格拆分成多个词
        words = segmented_text.split(' ') if segmented_text else []
        
        # 为每个词创建一行新的三元组
        for word in words:
            new_rows.append([col_c_value, word])  # COL_C 和 单个词组合成一行

    # 将新生成的行转换为 DataFrame
    new_df = pd.DataFrame(new_rows, columns=['COL_C', 'Element'])

    # 使用 tqdm 包裹分词列表，显示进度条
    tqdm.pandas(desc="Processing words")  # 设置进度条描述
    
    # 对每个词进行产业分类
    new_df['Industry'] = new_df['Element'].progress_apply(lambda x: classify_word(x))
    
    # 输出为新的 Excel 文件
    output_path = file_path.replace('.xlsx', '_classified.xlsx')
    new_df.to_excel(output_path, index=False)
    print(f"分类结果已保存到 {output_path}")

# 示例使用
file_path = r"E:\00people's commune_only_production.xlsx"  # 替换为你的文件路径
process_excel(file_path)

Processing words: 100%|██████████| 1217/1217 [1:59:26<00:00,  5.89s/it] 

分类结果已保存到 E:\00people's commune_only_production_classified.xlsx


In [ ]:
def classify_word(word):
    """使用 DeepSeek API 对单个词进行产业分类"""
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    # 分类提示词
    prompt = f"""
    你是一个专业的经济活动分类器。请将以下关于村落居民生产活动的词语分类：
    1. **农业**：直接涉及农作物种植、畜禽养殖、林产品采集等第一产业活动
    2. **工业**：涉及产品制造、加工、维修等第二产业活动
    3. **其他**：不属于以上两类的服务业或其他活动
    
    分类规则：
    - 农业类：粮食作物(如米、麦)、经济作物、畜禽养殖(养猪、养牛)、林产品(人参、药材)
    - 工业类：工厂生产(农具工厂)、设备维修(修配厂)、手工艺品制作
    - 其他：商业(小卖部)、旅游、教育、渔业(捕鱼)、运输等

    请只返回分类结果：农业、工业、其他
    需要分类的词语：{word}
    """

    payload = {
        "model": "deepseek-chat",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 10
    }

    try:
        response = requests.post(API_URL, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()["choices"][0]["message"]["content"].strip()
        
        if "农业" in result:
            return "农业"
        elif "工业" in result:
            return "工业"
        else:
            return "其他"
    
    except Exception as e:
        print(f"分类错误: {word} - {str(e)}")
        return "分类失败"

def process_excel(file_path):
    """加载 Excel 文件，拆分分词并进行产业分类"""
    # 加载 Excel 文件
    df = pd.read_excel(file_path)
    
    # 只提取 COL_C 和 segmented_text 列
    df_subset = df[['COL_C', 'segmented_text']]

    # 新的列表来保存处理后的数据
    new_rows = []

    # 遍历 DataFrame 中的每一行
    for index, row in df_subset.iterrows():
        col_c_value = row['COL_C']
        
        # 确保 segmented_text 是字符串类型，并处理 NaN 或其他类型
        segmented_text = str(row['segmented_text']) if pd.notna(row['segmented_text']) else ""
        
        # 将 segmented_text 按空格拆分成多个词
        words = segmented_text.split(' ') if segmented_text else []
        
        # 为每个词创建一行新的三元组
        for word in words:
            new_rows.append([col_c_value, word])  # COL_C 和 单个词组合成一行

    # 将新生成的行转换为 DataFrame
    new_df = pd.DataFrame(new_rows, columns=['COL_C', 'Element'])

    # 使用 tqdm 包裹分词列表，显示进度条
    tqdm.pandas(desc="Processing words")  # 设置进度条描述
    
    # 对每个词进行产业分类
    new_df['Industry'] = new_df['Element'].progress_apply(lambda x: classify_word(x))
    
    # 输出为新的 Excel 文件
    output_path = file_path.replace('.xlsx', '_classified.xlsx')
    new_df.to_excel(output_path, index=False)
    print(f"分类结果已保存到 {output_path}")

In [ ]:
# 示例使用
file_path = r"E:\00people's commune_only_production.xlsx"  # 替换为你的文件路径
process_excel(file_path)